Läser in alla regiondata för rumsbeläggning och gästnätter. Gör lite feature engineering

In [2]:
import pandas as pd

In [3]:
from pathlib import Path

print(Path.cwd())

/Users/nicklas.thegerstrom/ws/ml/AppliedAI/tourism_weather


In [4]:
from pathlib import Path

DATA_DIR = Path("/Users/nicklas.thegerstrom/ws/ml/AppliedAI/tourism_weather/data")

In [5]:

def load_folder_simple(folder_path, value_name):
   
    dfs = []

    for file in Path(folder_path).glob("*.xlsx"):
        df = pd.read_excel(file, skiprows=4)
        df.columns = ["month_name", "year", value_name]
        df["region"] = file.stem
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

In [6]:
guest_df = load_folder_simple(DATA_DIR / "guest_nights", "guest_nights")
occ_df = load_folder_simple(DATA_DIR / "occupancy", "occupancy_rate")

/opt/anaconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/anaconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/anaconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/anaconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/anaconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook con

In [7]:
month_map = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
    "Maj": 5, "Jun": 6, "Jul": 7, "Aug": 8,
    "Sep": 9, "Okt": 10, "Nov": 11, "Dec": 12
}

def add_date(df):
    df = df.copy()

    df["month"] = df["month_name"].map(month_map)
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

    df = df.dropna(subset=["year", "month"])

    df["date"] = pd.to_datetime(
        dict(
            year=df["year"].astype(int),
            month=df["month"].astype(int),
            day=1
        )
    )

    return df

In [8]:
guest_df = add_date(guest_df)
occ_df = add_date(occ_df)

In [9]:
print(guest_df.columns)
print(occ_df.columns)

Index(['month_name', 'year', 'guest_nights', 'region', 'month', 'date'], dtype='object')
Index(['month_name', 'year', 'occupancy_rate', 'region', 'month', 'date'], dtype='object')


In [10]:
df_t= guest_df.merge(
    occ_df,
    on=["region", "date"],
    how="inner"
)

In [11]:
df_t.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1230 entries, 0 to 1229
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   month_name_x    1230 non-null   object        
 1   year_x          1230 non-null   float64       
 2   guest_nights    1230 non-null   object        
 3   region          1230 non-null   object        
 4   month_x         1230 non-null   float64       
 5   date            1230 non-null   datetime64[ns]
 6   month_name_y    1230 non-null   object        
 7   year_y          1230 non-null   float64       
 8   occupancy_rate  1230 non-null   object        
 9   month_y         1230 non-null   float64       
dtypes: datetime64[ns](1), float64(4), object(5)
memory usage: 96.2+ KB


In [12]:
df_t = df_t.rename(columns={
    "year_x": "year",
    "month_x": "month"
})

df_t = df_t.drop(columns=[
    "month_name_x",
    "month_name_y",
    "year_y",
    "month_y"
])

In [13]:
df_t["guest_nights"] = pd.to_numeric(df_t["guest_nights"], errors="coerce")
df_t["occupancy_rate"] = pd.to_numeric(df_t["occupancy_rate"], errors="coerce")

In [14]:
df_t = df_t.dropna().sort_values(["region", "date"]).reset_index(drop=True)

In [15]:
df_t.info()
df_t.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1230 entries, 0 to 1229
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   year            1230 non-null   float64       
 1   guest_nights    1230 non-null   int64         
 2   region          1230 non-null   object        
 3   month           1230 non-null   float64       
 4   date            1230 non-null   datetime64[ns]
 5   occupancy_rate  1230 non-null   float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(1)
memory usage: 57.8+ KB


,year,guest_nights,region,month,date,occupancy_rate
0,2015.0,815341,dalarna,7.0,2015-07-01,0.554058
1,2015.0,459539,dalarna,8.0,2015-08-01,0.427013
2,2015.0,172431,dalarna,9.0,2015-09-01,0.327624
3,2015.0,154394,dalarna,10.0,2015-10-01,0.285888
4,2015.0,127696,dalarna,11.0,2015-11-01,0.280259


In [16]:
df_t["year"] = df_t["year"].astype(int)
df_t["month"] = df_t["month"].astype(int)

In [17]:
df_t.groupby("region").size()

region
dalarna           129
gotland           129
jamtland          129
norrbotten        129
riket              69
skane             129
stockholm         129
vasterbotten      129
vasternorrland    129
vastragotaland    129
dtype: int64

In [18]:
df_t["guest_lag1"] = df_t.groupby("region")["guest_nights"].shift(1)
df_t["guest_lag2"] = df_t.groupby("region")["guest_nights"].shift(2)
df_t["guest_lag12"] = df_t.groupby("region")["guest_nights"].shift(12)

df_t["occ_lag1"] = df_t.groupby("region")["occupancy_rate"].shift(1)

In [19]:
df_t[["region", "date", "guest_nights", "guest_lag1"]].head()

,region,date,guest_nights,guest_lag1
0,dalarna,2015-07-01,815341,NaN
1,dalarna,2015-08-01,459539,815341.0
2,dalarna,2015-09-01,172431,459539.0
3,dalarna,2015-10-01,154394,172431.0
4,dalarna,2015-11-01,127696,154394.0


In [20]:
df_t.isna().sum()

year                0
guest_nights        0
region              0
month               0
date                0
occupancy_rate      0
guest_lag1         10
guest_lag2         20
guest_lag12       120
occ_lag1           10
dtype: int64

In [21]:
import pandas as pd
import re
from pathlib import Path

def load_smhi_temp(path, region):
    rows = []

    with open(path, encoding="utf-8-sig") as f:
        for line in f:
            parts = line.strip().split(";")

            for i, value in enumerate(parts):
                value = value.strip()

                # hittar månad, t.ex. 2015-07
                if re.match(r"^\d{4}-\d{2}$", value):
                    if i + 1 < len(parts):
                        temp = parts[i + 1].replace(",", ".")
                        rows.append({
                            "region": region,
                            "date": pd.to_datetime(value),
                            "temp_mean": pd.to_numeric(temp, errors="coerce")
                        })

    weather = pd.DataFrame(rows)
    weather = weather.dropna().reset_index(drop=True)

    return weather

In [22]:
from pathlib import Path

weather_parts = []

for file in Path("data/smhi/temp").glob("*.csv"):
    region = file.stem
    weather_parts.append(load_smhi_temp(file, region))

weather_df = pd.concat(weather_parts, ignore_index=True)

In [23]:
weather_df["region"].unique()
weather_df.groupby("region").size()

region
dalarna           684
gotland           964
jamtland          983
lulea             973
skane             507
stockholm         351
sundsvall         939
umea              731
vastragotaland    432
dtype: int64

In [24]:
df_t["region"].unique()

array(['dalarna', 'gotland', 'jamtland', 'norrbotten', 'riket', 'skane',
       'stockholm', 'vasterbotten', 'vasternorrland', 'vastragotaland'],
      dtype=object)

In [25]:
weather_df["region"] = weather_df["region"].replace({
    "lulea": "norrbotten",
    "umea": "vasterbotten",
    "sundsvall": "vasternorrland"
})

In [26]:
set(df_t["region"].unique()) - set(weather_df["region"].unique())

{'riket'}

In [27]:
df_final = df_t.merge(
    weather_df,
    on=["region", "date"],
    how="left"
)

In [28]:
df_final.groupby("region")["temp_mean"].apply(lambda x: x.isna().sum())

region
dalarna            2
gotland            2
jamtland           2
norrbotten         2
riket             69
skane              7
stockholm          2
vasterbotten       2
vasternorrland     2
vastragotaland     4
Name: temp_mean, dtype: int64

In [29]:
df_model = df_final.dropna().reset_index(drop=True)

In [30]:
def load_smhi_daily_rain(path, region):
    rows = []

    with open(path, encoding="utf-8-sig") as f:
        for line in f:
            parts = [p.strip() for p in line.strip().split(";")]

            # SMHI regnfil:
            # 0 = från datetime
            # 1 = till datetime
            # 2 = representativt dygn
            # 3 = nederbörd
            if len(parts) >= 4:
                date = pd.to_datetime(parts[2], errors="coerce")
                rain = pd.to_numeric(parts[3].replace(",", "."), errors="coerce")

                if pd.notna(date) and pd.notna(rain):
                    rows.append({
                        "region": region,
                        "date": date,
                        "rain_mm": rain
                    })

    return pd.DataFrame(rows).reset_index(drop=True)

Aggregera regn från dag, till månad.

In [31]:
from pathlib import Path

rain_parts = []

for file in Path("data/smhi/regn").glob("*.csv"):
    region = file.stem
    rain_parts.append(load_smhi_daily_rain(file, region))

rain_df = pd.concat(rain_parts, ignore_index=True)

In [32]:
rain_df.head()
rain_df.groupby("region").size()

region
dalarna           60360
gotland           23427
jamtland          13825
norrbotten        25120
skane             11022
stockholm         26749
vasterbotten      19968
vasternorrland    19878
vastragotaland    16438
dtype: int64

In [33]:
rain_df["month"] = rain_df["date"].dt.to_period("M")

In [34]:
rain_df["month"] = rain_df["date"].dt.to_period("M")

rain_monthly = (
    rain_df
    .groupby(["region", "month"])
    .agg(
        rain_sum=("rain_mm", "sum"),
        rain_days=("rain_mm", lambda x: (x > 1).sum()),
        rain_mean=("rain_mm", "mean")
    )
    .reset_index()
)

rain_monthly["date"] = rain_monthly["month"].dt.to_timestamp()
rain_monthly = rain_monthly.drop(columns="month")

In [35]:
rain_monthly.head()

,region,rain_sum,rain_days,rain_mean,date
0,dalarna,98.4,12,3.174194,1860-01-01
1,dalarna,13.5,3,0.465517,1860-02-01
2,dalarna,27.1,5,0.874194,1860-03-01
3,dalarna,63.4,10,2.113333,1860-04-01
4,dalarna,55.3,8,1.783871,1860-05-01


In [36]:
rain_monthly = rain_monthly[
    (rain_monthly["date"] >= df_t["date"].min()) &
    (rain_monthly["date"] <= df_t["date"].max())
].copy()

In [37]:
rain_monthly.head()
rain_monthly.groupby("region").size()

region
dalarna           127
gotland           127
jamtland          127
norrbotten        127
skane             127
stockholm         127
vasterbotten      127
vasternorrland    127
vastragotaland    127
dtype: int64

In [38]:
rain_monthly.groupby("region")["date"].agg(["min", "max"])

,min,max
region,,
dalarna,2015-07-01,2026-01-01
gotland,2015-07-01,2026-01-01
jamtland,2015-07-01,2026-01-01
norrbotten,2015-07-01,2026-01-01
skane,2015-07-01,2026-01-01
stockholm,2015-07-01,2026-01-01
vasterbotten,2015-07-01,2026-01-01
vasternorrland,2015-07-01,2026-01-01
vastragotaland,2015-07-01,2026-01-01


In [39]:
df_final = df_final.merge(
    rain_monthly,
    on=["region", "date"],
    how="left"
)

In [40]:
df_final.groupby("region")[["rain_sum", "rain_days"]].apply(lambda x: x.isna().sum())

,rain_sum,rain_days
region,,
dalarna,2,2
gotland,2,2
jamtland,2,2
norrbotten,2,2
riket,69,69
skane,2,2
stockholm,2,2
vasterbotten,2,2
vasternorrland,2,2


In [41]:
df_model = df_final.dropna().reset_index(drop=True)

In [42]:
df_model.sample(20)

,year,guest_nights,region,month,date,occupancy_rate,guest_lag1,guest_lag2,guest_lag12,occ_lag1,temp_mean,rain_sum,rain_days,rain_mean
1001,2023,775945,vastragotaland,9,2023-09-01,0.621235,1528551.0,2535768.0,770534.0,0.652560,14.6,71.5,7.0,2.383333
923,2017,474600,vastragotaland,3,2017-03-01,0.559655,432032.0,421362.0,501897.0,0.533942,2.6,65.7,11.0,2.119355
155,2019,21230,gotland,11,2019-11-01,0.260014,37243.0,59664.0,21414.0,0.313537,5.3,77.5,14.0,2.583333
933,2018,435153,vastragotaland,1,2018-01-01,0.487301,450160.0,546780.0,421362.0,0.484198,0.1,104.2,14.0,3.361290
593,2018,1381040,stockholm,6,2018-06-01,0.706045,1321140.0,1101628.0,1354057.0,0.708510,17.3,33.9,7.0,1.130000
634,2021,1087841,stockholm,11,2021-11-01,0.587311,1060063.0,901861.0,408130.0,0.541276,3.7,28.5,5.0,0.950000
738,2020,53850,vasterbotten,12,2020-12-01,0.216465,59883.0,104268.0,88799.0,0.303268,0.5,134.1,24.0,4.325806
492,2019,347028,skane,3,2019-03-01,0.519148,289882.0,274232.0,354862.0,0.487657,5.0,114.7,14.0,3.700000
16,2017,166381,dalarna,11,2017-11-01,0.346076,149043.0,192619.0,153684.0,0.318078,1.1,61.2,11.0,2.040000
912,2025,80827,vasternorrland,11,2025-11-01,0.519953,98294.0,101898.0,78866.0,0.573254,-1.0,25.9,7.0,0.863333


FÄRDIG!!!

In [57]:
riket_occ = pd.read_excel(
    "data/occupancy/riket.xlsx",
    skiprows=5,
    header=None
)

riket_occ.columns = ["month_name", "year", "occupancy_rate"]

/opt/anaconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [59]:
riket_occ["month"] = riket_occ["month_name"].map(month_map)

riket_occ["year"] = pd.to_numeric(riket_occ["year"], errors="coerce")
riket_occ["occupancy_rate"] = pd.to_numeric(
    riket_occ["occupancy_rate"],
    errors="coerce"
)

riket_occ = riket_occ.dropna(subset=["year", "month", "occupancy_rate"])

In [60]:
riket_occ["date"] = pd.to_datetime(
    dict(
        year=riket_occ["year"].astype(int),
        month=riket_occ["month"].astype(int),
        day=1
    )
)

In [61]:
riket_occ = (
    riket_occ[["date", "occupancy_rate"]]
    .rename(columns={"occupancy_rate": "occupancy_rate_total"})
    .sort_values("date")
    .reset_index(drop=True)
)

In [65]:
riket_gn = pd.read_excel(
    "data/guest_nights/riket.xlsx",
    skiprows=5,
    header=None
)

riket_gn.columns = ["month_name", "year", "guest_nights"]

/opt/anaconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [66]:
riket_gn["month"] = riket_gn["month_name"].map(month_map)

riket_gn["year"] = pd.to_numeric(riket_gn["year"], errors="coerce")
riket_gn["guest_nights"] = pd.to_numeric(
    riket_gn["guest_nights"],
    errors="coerce"
)

riket_gn = riket_gn.dropna(subset=["year", "month", "guest_nights"])

In [67]:
riket_gn["date"] = pd.to_datetime(
    dict(
        year=riket_gn["year"].astype(int),
        month=riket_gn["month"].astype(int),
        day=1
    )
)

In [68]:
riket_gn = (
    riket_gn[["date", "guest_nights"]]
    .rename(columns={"guest_nights": "guest_nights_total"})
    .sort_values("date")
    .reset_index(drop=True)
)

In [70]:
df_model = df_model.merge(
    riket_gn,
    on="date",
    how="left"
)

In [71]:
df_model = df_model.merge(
    riket_occ,
    on="date",
    how="left"
)

In [72]:
df_model["share_of_total"] = (
    df_model["guest_nights"] / df_model["guest_nights_total"]
)

df_model["occupancy_vs_total"] = (
    df_model["occupancy_rate"] - df_model["occupancy_rate_total"]
)

In [73]:
df_model.to_csv("sweden_tourism_weather.csv", index=False)